# CCA-Arb on Colab
Runtime -> GPU. Needs a Hugging Face token for Llama (gated). Results persist to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/cca-arb-runs'

In [ ]:
!git clone https://github.com/AdityaShome/CCA-Arb.git /content/CCA-Arb
%cd /content/CCA-Arb
!pip install -q -r requirements.txt vllm
!git clone --depth 1 https://github.com/sierra-research/tau-bench third_party/tau-bench
!PYTHONUTF8=1 pip install -q --no-deps -e third_party/tau-bench
!python -m pytest tests -q -x

In [ ]:
# Serve one model per port. Edit --max-model-len / --gpu-memory-utilization for your GPU.
# T4 (16GB): use a 4-bit AWQ build or a smaller model, and --dtype half. L4/A100: two 7-8B models fit.
import subprocess, os
os.environ['HF_TOKEN']='PASTE_TOKEN'  # do not commit
subprocess.Popen('vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000 --gpu-memory-utilization 0.42 --max-model-len 8192 --dtype auto > /content/qwen.log 2>&1', shell=True)
subprocess.Popen('vllm serve meta-llama/Llama-3.1-8B-Instruct --port 8001 --gpu-memory-utilization 0.42 --max-model-len 8192 --dtype auto > /content/llama.log 2>&1', shell=True)

In [ ]:
import time, urllib.request
for port in (8000, 8001):
    for _ in range(120):
        try:
            urllib.request.urlopen(f'http://localhost:{port}/v1/models'); print(port,'up'); break
        except Exception: time.sleep(5)

In [ ]:
# Resumable: re-running skips finished (seed, task) pairs because --out lives on Drive.
!python scripts/run_experiment.py configs/CONFIG.yaml --out $OUT/RUN_NAME

In [ ]:
!python scripts/analyze.py $OUT/RUN_NAME --out $OUT/RUN_NAME/analysis.json